# 02 — QA Quality Analysis

Visualize answer length, citation accuracy, and response-time distributions from QA evaluation reports + live events.

In [1]:
import json
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().resolve()
if (REPO_ROOT / 'app').exists() is False:
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))

from app.analytics.analyze_qa import (
    analyze_qa_report,
    answer_length_distribution,
    citation_accuracy,
    top_questions,
    qa_event_time_breakdown,
)
from app.analytics.visualizer import plot_response_time_distribution

REPORT_PATH = REPO_ROOT / 'app/evaluation/reports/qa_eval_seed_report.json'
EVENTS_PATH = REPO_ROOT / 'app/storage/analytics/events.jsonl'
print('QA report exists:', REPORT_PATH.exists())
print('Events file exists:', EVENTS_PATH.exists())

QA report exists: True
Events file exists: True


## 1. Summary

In [2]:
report = json.loads(REPORT_PATH.read_text(encoding='utf-8'))
print(json.dumps(report['summary'], ensure_ascii=False, indent=2))

{
  "sample_count": 109,
  "evaluation_mode": "rule_based",
  "answer_pass_rate": 0.963302752293578,
  "citation_pass_rate": 1.0,
  "mean_answer_score": 0.963302752293578,
  "mean_citation_score": 1.0
}


## 2. Answer length distribution

In [3]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')
results = report['results']
df = pd.DataFrame([{ 'sample_id': r['sample_id'], 'answer_length': len(r.get('predicted_answer', '') or ''), 'answer_score': r.get('answer_evaluation', {}).get('score', 0.0), 'citation_score': r.get('citation_evaluation', {}).get('score', 0.0)} for r in results])
print(df.describe())
fig, ax = plt.subplots(figsize=(7,4))
sns.histplot(df['answer_length'], bins=30, kde=True, ax=ax, color='steelblue')
ax.set_title('Predicted Answer Length Distribution')
ax.set_xlabel('Characters')
fig

       answer_length  answer_score  citation_score
count     109.000000    109.000000           109.0
mean      285.073394      0.963303             1.0
std        83.631988      0.188886             0.0
min         8.000000      0.000000             1.0
25%       319.000000      1.000000             1.0
50%       320.000000      1.000000             1.0
75%       320.000000      1.000000             1.0
max       320.000000      1.000000             1.0


<Figure size 700x400 with 1 Axes>

## 3. Answer / citation score distribution

In [4]:
fig, ax = plt.subplots(figsize=(8,4))
df_melt = df.melt(id_vars='sample_id', value_vars=['answer_score','citation_score'], var_name='metric', value_name='score')
sns.boxplot(data=df_melt, x='metric', y='score', ax=ax)
ax.set_ylim(-0.05, 1.1)
fig

<Figure size 800x400 with 1 Axes>

## 4. Statistical test: are abstract questions easier than section ones?

In [5]:
from scipy import stats
abstract_scores = [r.get('answer_evaluation', {}).get('score', 0.0) for r in results if 'abstract' in r['sample_id']]
section_scores = [r.get('answer_evaluation', {}).get('score', 0.0) for r in results if 'section' in r['sample_id']]
print(f'abstract: n={len(abstract_scores)} mean={sum(abstract_scores)/max(1,len(abstract_scores)):.3f}')
print(f'section:  n={len(section_scores)} mean={sum(section_scores)/max(1,len(section_scores)):.3f}')
if abstract_scores and section_scores:
    t, p = stats.ttest_ind(abstract_scores, section_scores, equal_var=False)
    print(f'Welch t-test: t={t:.3f}, p={p:.4f} (significant if < 0.05)')

abstract: n=8 mean=1.000
section:  n=93 mean=1.000
Welch t-test: t=nan, p=nan (significant if < 0.05)


/home/chase/miniconda3/envs/research_agent/lib/python3.11/site-packages/scipy/stats/_axis_nan_policy.py:592: RuntimeWarning: Precision loss occurred in moment calculation due to catastrophic cancellation. This occurs when the data are nearly identical. Results may be unreliable.
  res = hypotest_fun_out(*samples, **kwds)


## 5. Live response time breakdown (if events.jsonl is populated)

In [6]:
if EVENTS_PATH.exists():
    breakdown = qa_event_time_breakdown(EVENTS_PATH)
    print(json.dumps(breakdown, ensure_ascii=False, indent=2))
else:
    print('Run real QA calls to populate live events.')

{
  "retrieval_time": {
    "count": 9,
    "mean": 1.4433869113353366,
    "p95": 0.015475360007258132,
    "median": 0.007492763004847802
  },
  "llm_time": {
    "count": 9,
    "mean": 13.613769512002667,
    "p95": 17.7400046360126,
    "median": 14.826706313004252
  },
  "total_time": {
    "count": 9,
    "mean": 15.057156423338004,
    "p95": 22.32969929400133,
    "median": 15.357789440997294
  }
}


## 6. Conclusions

- Answer length tightly clusters around the 320-char cap (artifact of seed dataset truncation).
- Citation accuracy is 1.0 by design (stub places gold citations at rank 1).
- Real differentiation will appear after wiring `evaluate_qa --use-live-pipeline`.